# Chapter 0: TL;DR — Fine-Tuning Phi-3 Mini into Yoda-speak

Godoy, *A Hands-On Guide to Fine-Tuning LLMs with PyTorch and Hugging Face*.

Kernel: **Finetuning_HF (.venv)**

In [4]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from transformers.utils import quantization_config


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32
)

repo_id = "microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    repo_id, device_map="cuda:0", quantization_config=bnb_config
)

Loading weights:   1%|          | 2/195 [00:02<03:37,  1.13s/it]/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 195/195 [00:37<00:00,  5.27it/s]


In [6]:
print(model.get_memory_footprint()/1e6)

2206.341504


In [7]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=

In [8]:
model = prepare_model_for_kbit_training(model)
config = LoraConfig(
  # the rank of the adapter, the lower the fewer parameters you'll need to train
  r=8,
  lora_alpha=16, # multiplier, usually 2*r
  bias="none",
  lora_dropout=0.05,
  task_type="CAUSAL_LM",
  # Newer models, such as Phi-3 at time of writing, may require
  # manually setting target modules
  target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'],
)
model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
        (layers): ModuleList(
          (0-31): 32 x Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (

In [9]:
print(model.get_memory_footprint()/1e6)

2651.074944


In [10]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Trainable parameters: {train_p/1e6:.2f}M')
print(f'Total parameters: {tot_p/1e6:.2f}M')
print(f'% of trainable parameters: {100*train_p/tot_p:.2f}%')

Trainable parameters: 12.58M
Total parameters: 3833.66M
% of trainable parameters: 0.33%


In [11]:
dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset

Dataset({
    features: ['sentence', 'translation', 'translation_extra'],
    num_rows: 720
})

In [12]:
dataset[0]

{'sentence': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'translation_extra': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [13]:
dataset = dataset.rename_column("sentence", "prompt")
dataset = dataset.rename_column("translation_extra", "completion")
dataset = dataset.remove_columns(["translation"])
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 720
})

In [14]:
dataset[0]

{'prompt': 'The birch canoe slid on the smooth planks.',
 'completion': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [15]:
messages = [
    {"role": "user", "content": dataset[0]['prompt']},
    {"role": "assistant", "content": dataset[0]['completion']}  
]

In [16]:
messages

[{'role': 'user', 'content': 'The birch canoe slid on the smooth planks.'},
 {'role': 'assistant',
  'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}]

In [17]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
tokenizer.chat_template

"{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'user' %}{{'<|user|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>\n' + message['content'] + '<|end|>\n'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% else %}{{ eos_token }}{% endif %}"

In [18]:
print(tokenizer.apply_chat_template(messages, tokenize=False))

<|user|>
The birch canoe slid on the smooth planks.<|end|>
<|assistant|>
On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>
<|endoftext|>


In [19]:
sft_config = SFTConfig(
  ## GROUP 1: Memory usage
  # These arguments will squeeze the most out of your GPU's RAM
  # Checkpointing
  gradient_checkpointing=True, # this saves a LOT of memory
  # Set this to avoid exceptions in newer versions of PyTorch
  gradient_checkpointing_kwargs={'use_reentrant': False},
  # Gradient Accumulation / Batch size
  # Actual batch (for updating) is same (1x) as micro-batch size
  gradient_accumulation_steps=1,
  # The initial (micro) batch size to start off with
  per_device_train_batch_size=16,
  # If batch size would cause OOM, halves its size until it works
  auto_find_batch_size=True,

  ## GROUP 2: Dataset-related
  max_length=64,
  # Dataset
  # packing a dataset means no padding is needed
  packing=True,
  ## GROUP 3: These are typical training parameters
  num_train_epochs=10,
  learning_rate=3e-4,
  # Optimizer
  # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
  optim='paged_adamw_8bit',
  ## GROUP 4: Logging parameters
  logging_steps=10,
  logging_dir='./logs',
  output_dir='./phi3-mini-yoda-adapter',
  report_to='none'
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset,
)

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported Flash Attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported Flash Attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

In [21]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [22]:
batch['input_ids'][0], batch['labels'][0]

(tensor([  319,  7055,   297,   278, 12164,  4225,   664,   304,  1248,   728,
         29889,  5531,   304,  1248,   728, 29892,   263,  7055,   297,   278,
         12164,  4225, 29889, 32000,   341,  1239,  5036,   338,   263,   270,
           728,  6766,   304,  4344, 29889, 29909,   270,   728,  6766,   304,
          4344, 29892,   286,  1239,  5036,   338, 29889, 32000,   450, 15795,
           289,   929,   911,  2996,  4822,   515,   278,  7205, 29889, 29907,
           420,  4822,   515,   278,  7205, 29892,   278, 15795,   289,   929,
           911,  1258, 29889,  3869, 29892, 22157,  1758,  4317, 29889, 32000,
          5254,   278,   282,  1934,   322,   409, 29893,   263,  2826,   373,
           278, 16779, 29889, 10923,   278,   282,  1934, 29892,   366,  1818,
         29892,   322,   263,  2826,   373,   278, 16779, 29892,   409, 29893,
         29889, 32000,   450, 19756,   526, 12474,   322,  2989,   310, 15301,
         12169, 29889, 29940,  2936,   322,  2989,  

In [23]:
trainer.train()

Step,Training Loss
10,1.806561
20,0.443099
30,0.315115
40,0.302193
50,0.233703
60,0.224351
70,0.187783
80,0.152549
90,0.133943
100,0.116759


TrainOutput(global_step=210, training_loss=0.20312396393024496, metrics={'train_runtime': 369.6656, 'train_samples_per_second': 8.873, 'train_steps_per_second': 0.568, 'total_flos': 4276013292748800.0, 'train_loss': 0.20312396393024496, 'epoch': 10.0})

In [24]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [{"role": "user", "content": sentence}] 
    prompt = tokenizer.apply_chat_template(
        converted_sample, tokenize=False, add_generation_prompt=True
    )
    return prompt

In [25]:
sentence = 'The Force is strong in you!'
prompt = gen_prompt(tokenizer, sentence)    
print(prompt)

<|user|>
The Force is strong in you!<|end|>
<|assistant|>



In [31]:
def generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=64,
    skip_special_tokens=False,
):
    tokenized_input = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(model.device)

    model.eval()

    gen_output = model.generate(
        **tokenized_input,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=max_new_tokens,
    )

    output = tokenizer.batch_decode(
        gen_output,
        skip_special_tokens=skip_special_tokens,
    )

    return output[0]

In [32]:
print(generate(model, tokenizer, prompt))

<|user|> The Force is strong in you!<|end|><|assistant|> Strong in you, the Force is!<|end|><|endoftext|>


In [33]:
trainer.save_model('local-phi3-mini-yoda-adapter')

In [34]:
from huggingface_hub import login
login()

In [35]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/ankitanand9/phi3-mini-yoda-adapter/commit/9ee692709d75a0da77a25a9509fdc3bc336c0332', commit_message='End of training', commit_description='', oid='9ee692709d75a0da77a25a9509fdc3bc336c0332', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ankitanand9/phi3-mini-yoda-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='ankitanand9/phi3-mini-yoda-adapter'), pr_revision=None, pr_num=None)